In [ ]:
!zip -FF /content/veremi1.zip --out /content/veremi_fixed.zip
!unzip -q -o /content/veremi_fixed.zip -d /content/veremi
!find /content/veremi -type f | head -20
!find /content/veremi -type f | wc -l

Fix archive (-FF) - salvage what can
	zip warning: Missing end (EOCDR) signature - either this archive
                     is not readable or the end is damaged
Is this a single-disk archive?  (y/n): y
  Assuming single-disk archive
Scanning for entries...
 copying: VeReMi_25200_28800_2022-9-11_20:45:32/  (0 bytes)
 copying: VeReMi_25200_28800_2022-9-11_20:45:32/traceJSON-9-7-A0-25202-7.json  (31315 bytes)
 copying: VeReMi_25200_28800_2022-9-11_20:45:32/traceGroundTruthJSON-7.json  (31694341 bytes)
 copying: VeReMi_25200_28800_2022-9-11_20:45:32/traceJSON-15-13-A1-25207-7.json  (10613 bytes)
 copying: VeReMi_25200_28800_2022-9-11_20:45:32/traceJSON-21-19-A0-25208-7.json  (86991 bytes)
 copying: VeReMi_25200_28800_2022-9-11_20:45:32/traceJSON-27-25-A0-25208-7.json  (25694 bytes)
 copying: VeReMi_25200_28800_2022-9-11_20:45:32/traceJSON-33-31-A1-25210-7.json  (71565 bytes)
 copying: VeReMi_25200_28800_2022-9-11_20:45:32/traceJSON-39-37-A0-25211-7.json  (19164 bytes)
 copying: VeReMi_252

In [25]:
import json, re, glob, os
import pandas as pd, numpy as np

SIM = sorted(glob.glob('/content/veremi/VeReMi_*'))[0]
pat = re.compile(r'traceJSON-(\d+)-(\d+)-A(\d)-(\d+)-(\d+)\.json')

recv_rows, own_rows, bad = [], [], []
for f in glob.glob(f'{SIM}/traceJSON-*.json'):
    m = pat.search(os.path.basename(f))
    if not m:
        continue
    rid, attacker_flag = int(m[1]), int(m[3])
    r_tmp, o_tmp = [], []
    try:
        with open(f) as fh:
            for line in fh:
                o = json.loads(line)
                if o['type'] == 3:
                    o['receiver'] = rid; o['rcv_is_attacker'] = attacker_flag
                    r_tmp.append(o)
                elif o['type'] == 2:
                    o['vehicle'] = rid
                    o_tmp.append(o)
        recv_rows += r_tmp; own_rows += o_tmp      # only keep fully-read files
    except Exception as e:
        bad.append((os.path.basename(f), str(e)[:80]))

print(f"Files OK: {len(glob.glob(f'{SIM}/traceJSON-*.json')) - len(bad)} | skipped (corrupt): {len(bad)}")

def flatten(df):
    for c in ['pos', 'pos_noise', 'spd', 'spd_noise', 'acl', 'acl_noise', 'hed', 'hed_noise']:
        if c in df.columns:
            arr = np.array(df[c].tolist(), dtype='float32')
            df[f'{c}_x'], df[f'{c}_y'] = arr[:, 0], arr[:, 1]
            df.drop(columns=c, inplace=True)
    return df

msgs = flatten(pd.DataFrame(recv_rows))     # received BSMs (training data)
own  = flatten(pd.DataFrame(own_rows))      # each vehicle's own GPS (for RSU participation)
del recv_rows, own_rows

gt_file = glob.glob(f'{SIM}/traceGroundTruthJSON-*.json')[0]
gt = flatten(pd.read_json(gt_file, lines=True))
print("GT columns:", gt.columns.tolist())

Files OK: 1026 | skipped (corrupt): 1
GT columns: ['type', 'sendTime', 'sender', 'senderPseudo', 'messageID', 'pos_x', 'pos_y', 'pos_noise_x', 'pos_noise_y', 'spd_x', 'spd_y', 'spd_noise_x', 'spd_noise_y', 'acl_x', 'acl_y', 'acl_noise_x', 'acl_noise_y', 'hed_x', 'hed_y', 'hed_noise_x', 'hed_noise_y']


In [ ]:
# Build attacker maps from filenames, using both ID positions
by_first, by_second = {}, {}
for f in glob.glob(f'{SIM}/traceJSON-*.json'):
    m = pat.search(os.path.basename(f))
    if m:
        by_first[int(m[1])] = int(m[3])
        by_second[int(m[2])] = int(m[3])

senders = set(msgs['sender'].unique())
match1, match2 = len(senders & set(by_first)), len(senders & set(by_second))
print(f"Senders: {len(senders)} | matched by vehicle-ID: {match1} | by module-ID: {match2}")
att_map = by_first if match1 >= match2 else by_second

msgs['sender_is_attacker'] = msgs['sender'].map(att_map)
print("Messages with unknown sender status:", msgs['sender_is_attacker'].isna().sum())
msgs = msgs.dropna(subset=['sender_is_attacker'])
msgs['y_bin'] = msgs['sender_is_attacker'].astype('int64')

# Attach true sender positions from ground truth (for reconstruction error later)
gt_small = gt[['messageID', 'pos_x', 'pos_y']].rename(
    columns={'pos_x': 'true_pos_x', 'pos_y': 'true_pos_y'})
msgs = msgs.merge(gt_small, on='messageID', how='left')

# Sanity check: attacker messages should deviate from ground truth more
msgs['pos_dev'] = np.hypot(msgs['pos_x'] - msgs['true_pos_x'], msgs['pos_y'] - msgs['true_pos_y'])
print("\nPosition deviation from GT (m), by label:")
print(msgs.groupby('y_bin')['pos_dev'].describe().round(2))

print("\nMessages:", msgs.shape)
print("Label balance:\n", msgs['y_bin'].value_counts(normalize=True).round(3))
print("\nReceivers (FL clients):", msgs['receiver'].nunique())
print("Messages per receiver:", msgs.groupby('receiver').size().describe().round(0).to_dict())
print("Misbehaviour ratio per receiver:", msgs.groupby('receiver')['y_bin'].mean().describe().round(3).to_dict())
print("\nPseudonyms per sender:", msgs.groupby('sender')['senderPseudo'].nunique().describe().round(1).to_dict())
print("Own-GPS rows:", own.shape)

msgs.to_parquet(f"{OUT}/sim7_msgs.parquet")
own.to_parquet(f"{OUT}/sim7_own.parquet")
gt.to_parquet(f"{OUT}/sim7_gt.parquet")

Senders: 1107 | matched by vehicle-ID: 1027 | by module-ID: 0
Messages with unknown sender status: 10484

Position deviation from GT (m), by label:
          count    mean     std   min     25%     50%     75%      max
y_bin                                                                 
0      424879.0    0.00    0.00  0.00    0.00    0.00    0.00     0.00
1      172102.0  746.51  340.23  4.35  474.34  739.69  996.08  1658.65

Messages: (596981, 29)
Label balance:
 y_bin
0    0.712
1    0.288
Name: proportion, dtype: float64

Receivers (FL clients): 1026
Messages per receiver: {'count': 1026.0, 'mean': 582.0, 'std': 387.0, 'min': 6.0, '25%': 269.0, '50%': 514.0, '75%': 822.0, 'max': 2028.0}
Misbehaviour ratio per receiver: {'count': 1026.0, 'mean': 0.285, 'std': 0.111, 'min': 0.0, '25%': 0.21, '50%': 0.284, '75%': 0.356, 'max': 0.714}

Pseudonyms per sender: {'count': 1027.0, 'mean': 1.0, 'std': 0.0, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 1.0}
Own-GPS rows: (104355, 1

In [26]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

import os, copy, random, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, accuracy_score

OUT = "/content/drive/MyDrive/capstone/outputs"
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", dev)

Mounted at /content/drive
Device: cuda


In [ ]:
msgs = pd.read_parquet(f"{OUT}/sim7_msgs.parquet")
own  = pd.read_parquet(f"{OUT}/sim7_own.parquet")
print(msgs.shape, own.shape)

(596981, 29) (104355, 19)


In [ ]:
d = msgs.copy()
d = d.sort_values(['receiver', 'sender', 'rcvTime']).reset_index(drop=True)

g = d.groupby(['receiver', 'sender'], sort=False)
d['dt'] = g['rcvTime'].diff()
dx, dy = g['pos_x'].diff(), g['pos_y'].diff()
d['dist'] = np.hypot(dx, dy)
d['spd_mag'] = np.hypot(d.spd_x, d.spd_y)
d['implied_spd'] = d['dist'] / d['dt'].clip(lower=1e-3)
d['spd_err'] = (d['implied_spd'] - d['spd_mag']).abs()
d['delay'] = d['rcvTime'] - d['sendTime']

own_s = (own[['vehicle', 'rcvTime', 'pos_x', 'pos_y']]
         .rename(columns={'vehicle': 'receiver', 'pos_x': 'rx_x', 'pos_y': 'rx_y'})
         .sort_values('rcvTime'))
d = pd.merge_asof(d.sort_values('rcvTime'), own_s, on='rcvTime', by='receiver', direction='backward')
d['rx_dist'] = np.hypot(d.pos_x - d.rx_x, d.pos_y - d.rx_y)

d = d.dropna(subset=['dt', 'rx_dist']).reset_index(drop=True)
for c in ['dt', 'dist', 'implied_spd', 'spd_err', 'rx_dist']:
    d[c] = np.log1p(d[c].clip(lower=0))

FEATS = ['pos_x', 'pos_y', 'spd_x', 'spd_y', 'acl_x', 'acl_y', 'hed_x', 'hed_y',
         'pos_noise_x', 'pos_noise_y', 'spd_noise_x', 'spd_noise_y',
         'dt', 'dist', 'spd_mag', 'implied_spd', 'spd_err', 'delay', 'rx_dist']
d[FEATS] = d[FEATS].astype('float32')
print(d.shape, "| label balance:", d['y_bin'].value_counts(normalize=True).round(3).to_dict())
d.to_parquet(f"{OUT}/sim7_features.parquet")

(565997, 38) | label balance: {0: 0.712, 1: 0.288}


In [ ]:
d = msgs.copy()
d = d.sort_values(['receiver', 'sender', 'rcvTime']).reset_index(drop=True)

g = d.groupby(['receiver', 'sender'], sort=False)
d['dt'] = g['rcvTime'].diff()
dx, dy = g['pos_x'].diff(), g['pos_y'].diff()
d['dist'] = np.hypot(dx, dy)
d['spd_mag'] = np.hypot(d.spd_x, d.spd_y)
d['implied_spd'] = d['dist'] / d['dt'].clip(lower=1e-3)
d['spd_err'] = (d['implied_spd'] - d['spd_mag']).abs()
d['delay'] = d['rcvTime'] - d['sendTime']

own_s = (own[['vehicle', 'rcvTime', 'pos_x', 'pos_y']]
         .rename(columns={'vehicle': 'receiver', 'pos_x': 'rx_x', 'pos_y': 'rx_y'})
         .sort_values('rcvTime'))
d = pd.merge_asof(d.sort_values('rcvTime'), own_s, on='rcvTime', by='receiver', direction='backward')
d['rx_dist'] = np.hypot(d.pos_x - d.rx_x, d.pos_y - d.rx_y)

d = d.dropna(subset=['dt', 'rx_dist']).reset_index(drop=True)
for c in ['dt', 'dist', 'implied_spd', 'spd_err', 'rx_dist']:
    d[c] = np.log1p(d[c].clip(lower=0))

FEATS = ['pos_x', 'pos_y', 'spd_x', 'spd_y', 'acl_x', 'acl_y', 'hed_x', 'hed_y',
         'pos_noise_x', 'pos_noise_y', 'spd_noise_x', 'spd_noise_y',
         'dt', 'dist', 'spd_mag', 'implied_spd', 'spd_err', 'delay', 'rx_dist']
d[FEATS] = d[FEATS].astype('float32')
print(d.shape, "| label balance:", d['y_bin'].value_counts(normalize=True).round(3).to_dict())
d.to_parquet(f"{OUT}/sim7_features.parquet")

(565997, 38) | label balance: {0: 0.712, 1: 0.288}


In [ ]:
R, T, YCOL = 'receiver', 'rcvTime', 'y_bin'
MIN_SAMPLES = 200
counts = d[R].value_counts()
d = d[d[R].isin(counts[counts >= MIN_SAMPLES].index)].copy()

d = d.sort_values([R, T])
d['rank'] = d.groupby(R).cumcount()
d['n'] = d.groupby(R)[R].transform('size')
d['is_train'] = d['rank'] < 0.8 * d['n']

scaler = StandardScaler().fit(d.loc[d.is_train, FEATS])
d[FEATS] = scaler.transform(d[FEATS]).astype('float32')

clients = {}
for cid, grp in d.groupby(R):
    tr, te = grp[grp.is_train], grp[~grp.is_train]
    clients[cid] = dict(Xtr=torch.tensor(tr[FEATS].values), ytr=torch.tensor(tr[YCOL].values),
                        Xte=torch.tensor(te[FEATS].values), yte=torch.tensor(te[YCOL].values))
cids = list(clients)
Xte_all = torch.cat([c['Xte'] for c in clients.values()])
yte_all = torch.cat([c['yte'] for c in clients.values()])

sizes = np.array([len(c['ytr']) for c in clients.values()])
mis = np.array([c['ytr'].float().mean().item() for c in clients.values()])
print(f"Clients: {len(cids)} | train size min/median/max: {sizes.min()}/{int(np.median(sizes))}/{sizes.max()}")
print(f"Misbehaviour ratio across clients min/median/max: {mis.min():.2f}/{np.median(mis):.2f}/{mis.max():.2f}")

Clients: 839 | train size min/median/max: 163/453/1579
Misbehaviour ratio across clients min/median/max: 0.00/0.29/0.67


In [ ]:
N_FEAT, N_CLS = len(FEATS), int(d[YCOL].nunique())

class MLP(nn.Module):
    def __init__(self, d_in, h=64, c=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, h), nn.ReLU(),
                                 nn.Linear(h, h), nn.ReLU(),
                                 nn.Linear(h, c))
    def forward(self, x): return self.net(x)

@torch.no_grad()
def evaluate(model, X, y, bs=8192):
    model.eval(); preds = []
    for i in range(0, len(X), bs):
        preds.append(model(X[i:i+bs].to(dev)).argmax(1).cpu())
    p = torch.cat(preds).numpy()
    return accuracy_score(y, p), f1_score(y, p, average='macro')

Xtr_all = torch.cat([c['Xtr'] for c in clients.values()])
ytr_all = torch.cat([c['ytr'] for c in clients.values()])
w = torch.tensor(1.0 / np.bincount(ytr_all.numpy()), dtype=torch.float32)
w = (w / w.sum() * N_CLS).to(dev)
lossf = nn.CrossEntropyLoss(weight=w)

torch.manual_seed(0)
cent = MLP(N_FEAT, c=N_CLS).to(dev)
opt = torch.optim.Adam(cent.parameters(), lr=1e-3)
for ep in range(5):
    cent.train(); perm = torch.randperm(len(Xtr_all))
    for i in range(0, len(perm), 1024):
        idx = perm[i:i+1024]
        opt.zero_grad(); lossf(cent(Xtr_all[idx].to(dev)), ytr_all[idx].to(dev)).backward(); opt.step()
    acc, f1 = evaluate(cent, Xte_all, yte_all)
    print(f"Centralized epoch {ep+1}: acc={acc:.4f} macroF1={f1:.4f}")

Centralized epoch 1: acc=0.9944 macroF1=0.9931
Centralized epoch 2: acc=0.9957 macroF1=0.9947
Centralized epoch 3: acc=0.9975 macroF1=0.9969
Centralized epoch 4: acc=0.9987 macroF1=0.9984
Centralized epoch 5: acc=0.9991 macroF1=0.9989


In [ ]:
ROUNDS, K, LOCAL_EPOCHS, BS, LR = 50, 20, 1, 64, 0.05

def local_train(gmodel, X, y):
    m = copy.deepcopy(gmodel); m.train()
    opt = torch.optim.SGD(m.parameters(), lr=LR)
    X, y = X.to(dev), y.to(dev)
    for _ in range(LOCAL_EPOCHS):
        perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), BS):
            idx = perm[i:i+BS]
            opt.zero_grad(); lossf(m(X[idx]), y[idx]).backward(); opt.step()
    return {k: v.detach().clone() for k, v in m.state_dict().items()}, len(X)

def fedavg(updates):
    total = sum(n for _, n in updates)
    return {k: sum(sd[k] * (n / total) for sd, n in updates) for k in updates[0][0]}

random.seed(0); torch.manual_seed(0)
gmodel = MLP(N_FEAT, c=N_CLS).to(dev)
log, participation = [], []
t0 = time.time()
for r in range(ROUNDS):
    sel = random.sample(cids, min(K, len(cids)))
    participation.append(sel)
    updates = [local_train(gmodel, clients[c]['Xtr'], clients[c]['ytr']) for c in sel]
    gmodel.load_state_dict(fedavg(updates))
    if (r + 1) % 5 == 0:
        acc, f1 = evaluate(gmodel, Xte_all, yte_all)
        log.append(dict(round=r+1, acc=acc, f1=f1, secs=time.time()-t0))
        print(f"Round {r+1}: acc={acc:.4f} macroF1={f1:.4f} ({time.time()-t0:.0f}s)")

pd.DataFrame(log).to_csv(f"{OUT}/fedavg_baseline.csv", index=False)
torch.save(gmodel.state_dict(), f"{OUT}/fedavg_baseline.pt")

Round 5: acc=0.9738 macroF1=0.9684 (2s)
Round 10: acc=0.9851 macroF1=0.9817 (4s)
Round 15: acc=0.9836 macroF1=0.9799 (6s)
Round 20: acc=0.9868 macroF1=0.9838 (7s)
Round 25: acc=0.9869 macroF1=0.9839 (8s)
Round 30: acc=0.9875 macroF1=0.9847 (9s)
Round 35: acc=0.9874 macroF1=0.9845 (11s)
Round 40: acc=0.9879 macroF1=0.9851 (12s)
Round 45: acc=0.9881 macroF1=0.9853 (13s)
Round 50: acc=0.9876 macroF1=0.9848 (15s)


In [27]:
import torch.nn.functional as F
from scipy.optimize import linear_sum_assignment

POS = [FEATS.index('pos_x'), FEATS.index('pos_y')]
mu = torch.tensor(scaler.mean_, dtype=torch.float32)
sd = torch.tensor(scaler.scale_, dtype=torch.float32)
def unscale(X): return X.detach().cpu() * sd + mu
pos_mean = mu[POS].numpy()                     # "guess the average position" baseline

def true_grads(model, X, y):
    loss = lossf(model(X), y)
    return [g.detach() for g in torch.autograd.grad(loss, model.parameters())]

def invert(model, tgt, y, n_feat, iters=3000, lr=0.05, mode='cos'):
    x = torch.randn(len(y), n_feat, device=dev, requires_grad=True)
    opt = torch.optim.Adam([x], lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, [iters // 2, 3 * iters // 4], 0.1)
    for _ in range(iters):
        opt.zero_grad()
        dg = torch.autograd.grad(lossf(model(x), y), model.parameters(), create_graph=True)
        if mode == 'cos':
            rec = sum(1 - F.cosine_similarity(a.flatten(), b.flatten(), dim=0) for a, b in zip(dg, tgt))
        else:
            rec = sum(((a - b) ** 2).sum() for a, b in zip(dg, tgt))
        rec.backward(); opt.step(); sched.step()
    return x.detach(), rec.item()

def pos_error(x_hat, x_true):
    """Mean position error in metres, after optimally matching reconstructed rows to real rows."""
    P = unscale(x_hat)[:, POS].numpy(); Q = unscale(x_true)[:, POS].numpy()
    C = np.linalg.norm(P[:, None, :] - Q[None, :, :], axis=-1)
    r, c = linear_sum_assignment(C)
    return C[r, c].mean()

In [28]:
gmodel.eval()
for p in gmodel.parameters(): p.requires_grad_(True)
train_rows = d[d.is_train]
rng = np.random.default_rng(0)
test_cids = rng.choice(cids, 5, replace=False)

results = []
t0 = time.time()
for B in [1, 2, 4, 8, 16]:
    for cid in test_cids:
        rows = train_rows[train_rows[R] == cid].sample(B, random_state=0)
        X = torch.tensor(rows[FEATS].values, device=dev)
        y = torch.tensor(rows[YCOL].values, device=dev)
        tgt = true_grads(gmodel, X, y)
        x_hat, mloss = invert(gmodel, tgt, y, N_FEAT)

        err = pos_error(x_hat, X)
        base = np.linalg.norm(unscale(X)[:, POS].numpy() - pos_mean, axis=1).mean()
        rx_hat = unscale(x_hat)[:, POS].numpy().mean(0)      # estimate of the client's own location
        rx_true = rows[['rx_x', 'rx_y']].values.mean(0)
        results.append(dict(B=B, client=cid, pos_err_m=err, baseline_m=base,
                            client_loc_err_m=np.linalg.norm(rx_hat - rx_true), match_loss=mloss))
    print(f"B={B} done ({time.time()-t0:.0f}s)")

res = pd.DataFrame(results)
print(res.groupby('B')[['pos_err_m', 'baseline_m', 'client_loc_err_m', 'match_loss']].median().round(2))
res.to_csv(f"{OUT}/inversion_fedsgd_v1.csv", index=False)

B=1 done (80s)
B=2 done (158s)
B=4 done (246s)
B=8 done (326s)
B=16 done (406s)
     pos_err_m  baseline_m  client_loc_err_m  match_loss
B                                                       
1     0.040000  452.769989        143.339996        0.00
2     1.480000  463.450012        173.240005        0.00
4   283.700012  460.380005        301.209991        0.05
8   269.820007  479.190002        189.839996        0.03
16  370.260010  491.049988        183.110001        0.03


In [29]:
from scipy.optimize import least_squares
RXD = FEATS.index('rx_dist')

def locate_client(x_rows):
    U = unscale(x_rows).numpy()
    S = U[:, POS]                              # reconstructed sender positions
    dist = np.expm1(U[:, RXD]).clip(min=0)     # undo the log1p from feature engineering
    if len(S) < 3:
        return S.mean(0)                       # trilateration needs >= 3 messages
    f = lambda r: np.linalg.norm(S - r, axis=1) - dist
    return least_squares(f, S.mean(0)).x

def invert_best(model, tgt, y, n_feat, mode, restarts):
    best = None
    for _ in range(restarts):
        x_hat, ml = invert(model, tgt, y, n_feat, mode=mode)
        if best is None or ml < best[1]:
            best = (x_hat, ml)
    return best

In [30]:
results2 = []
t0 = time.time()
for mode in ['cos', 'l2']:
    for B in [1, 2, 4, 8, 16]:
        restarts = 1 if B <= 2 else 2
        for cid in test_cids:
            rows = train_rows[train_rows[R] == cid].sample(B, random_state=0)
            X = torch.tensor(rows[FEATS].values, device=dev)
            y = torch.tensor(rows[YCOL].values, device=dev)
            tgt = true_grads(gmodel, X, y)
            x_hat, mloss = invert_best(gmodel, tgt, y, N_FEAT, mode, restarts)

            rx_true = rows[['rx_x', 'rx_y']].values.mean(0)
            results2.append(dict(
                mode=mode, B=B, client=cid,
                pos_err_m=pos_error(x_hat, X),
                baseline_m=np.linalg.norm(unscale(X)[:, POS].numpy() - pos_mean, axis=1).mean(),
                loc_err_trilat_m=np.linalg.norm(locate_client(x_hat) - rx_true),
                loc_err_trilat_TRUEDATA_m=np.linalg.norm(locate_client(X) - rx_true),
                match_loss=mloss))
        print(f"{mode} B={B} done ({time.time()-t0:.0f}s)")

res2 = pd.DataFrame(results2)
print(res2.groupby(['mode', 'B'])[['pos_err_m', 'baseline_m', 'loc_err_trilat_m',
                                   'loc_err_trilat_TRUEDATA_m', 'match_loss']].median().round(2))
res2.to_csv(f"{OUT}/inversion_fedsgd_v2.csv", index=False)

cos B=1 done (80s)
cos B=2 done (159s)
cos B=4 done (314s)
cos B=8 done (467s)
cos B=16 done (622s)
l2 B=1 done (678s)
l2 B=2 done (738s)
l2 B=4 done (849s)
l2 B=8 done (960s)
l2 B=16 done (1073s)
          pos_err_m  baseline_m  loc_err_trilat_m  loc_err_trilat_TRUEDATA_m  \
mode B                                                                         
cos  1     0.030000  452.769989            143.17                     143.37   
     2     0.260000  463.450012            173.25                     173.26   
     4   266.779999  460.380005            334.41                      85.03   
     8   320.890015  479.190002            456.66                      81.14   
     16  281.799988  491.049988            333.31                     104.76   
l2   1   394.850006  452.769989            342.82                     143.37   
     2   782.679993  463.450012            689.68                     173.26   
     4   424.779999  460.380005            502.22                      85.03   
   

In [31]:
lo = torch.tensor(train_rows[FEATS].min().values, device=dev)
hi = torch.tensor(train_rows[FEATS].max().values, device=dev)

def sample_window(cid, B, seed):
    g = train_rows[train_rows[R] == cid]               # already time-ordered per client
    start = np.random.default_rng(seed).integers(0, len(g) - B)
    return g.iloc[start:start + B]

def invert_v3(model, tgt, y, n_feat, iters=8000, lr=0.1, seed=0):
    torch.manual_seed(seed)
    x = (0.1 * torch.randn(len(y), n_feat, device=dev)).requires_grad_(True)   # start near the data mean (0 after scaling)
    opt = torch.optim.Adam([x], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, iters)
    for _ in range(iters):
        opt.zero_grad()
        dg = torch.autograd.grad(lossf(model(x), y), model.parameters(), create_graph=True)
        rec = sum(1 - F.cosine_similarity(a.flatten(), b.flatten(), dim=0) for a, b in zip(dg, tgt))
        rec.backward(); opt.step(); sched.step()
        with torch.no_grad():
            x.data = torch.max(torch.min(x.data, hi), lo)                      # stay within valid feature ranges
    return x.detach(), rec.item()

results3 = []
t0 = time.time()
for B in [2, 4, 8, 16]:
    for cid in test_cids:
        rows = sample_window(cid, B, seed=int(cid) % 1000)
        X = torch.tensor(rows[FEATS].values, device=dev)
        y = torch.tensor(rows[YCOL].values, device=dev)
        tgt = true_grads(gmodel, X, y)

        best = None
        for s in range(2):                                                    # 2 restarts, keep the best match
            x_hat, ml = invert_v3(gmodel, tgt, y, N_FEAT, seed=s)
            if best is None or ml < best[1]: best = (x_hat, ml)
        x_hat, ml = best

        rx_true = rows[['rx_x', 'rx_y']].values.mean(0)
        results3.append(dict(
            B=B, client=cid,
            window_s=rows['rcvTime'].max() - rows['rcvTime'].min(),
            pos_err_m=pos_error(x_hat, X),
            baseline_m=np.linalg.norm(unscale(X)[:, POS].numpy() - pos_mean, axis=1).mean(),
            loc_err_trilat_m=np.linalg.norm(locate_client(x_hat) - rx_true),
            loc_err_trilat_TRUEDATA_m=np.linalg.norm(locate_client(X) - rx_true),
            match_loss=ml))
    print(f"B={B} done ({time.time()-t0:.0f}s)")

res3 = pd.DataFrame(results3)
print(res3.groupby('B')[['window_s', 'pos_err_m', 'baseline_m', 'loc_err_trilat_m',
                         'loc_err_trilat_TRUEDATA_m', 'match_loss']].median().round(2))
res3.to_csv(f"{OUT}/inversion_fedsgd_v3.csv", index=False)

B=2 done (419s)
B=4 done (840s)
B=8 done (1263s)
B=16 done (1689s)
    window_s   pos_err_m  baseline_m  loc_err_trilat_m  \
B                                                        
2       0.27    0.000000  516.510010            303.21   
4       0.64   62.139999  540.450012             70.68   
8       1.00  256.160004  493.739990            250.42   
16      2.00  304.760010  515.659973            410.15   

    loc_err_trilat_TRUEDATA_m  match_loss  
B                                          
2                      303.21         0.0  
4                        0.00         0.0  
8                        1.05         0.0  
16                       2.78         0.0  


In [32]:
def analytic_candidates(tgt, eps=1e-10):
    gW, gb = tgt[0], tgt[1]                 # first-layer weight and bias gradients
    mask = gb.abs() > eps
    return (gW[mask] / gb[mask, None]).cpu().numpy()   # one candidate input per active neuron

def recover_rows(C, tol=1e-3):
    """Candidates that several neurons agree on are exactly recovered inputs."""
    used = np.zeros(len(C), bool); rec = []
    for i in range(len(C)):
        if used[i]: continue
        close = np.where(np.linalg.norm(C - C[i], axis=1) < tol)[0]
        used[close] = True
        if len(close) >= 2: rec.append(C[close].mean(0))
    return np.array(rec)

results4 = []
rng2 = np.random.default_rng(1)
t0 = time.time()
for B in [2, 4, 8, 16, 32, 64]:
    for cid in rng2.choice(cids, 20, replace=False):
        rows = sample_window(cid, B, seed=int(cid) % 1000 + B)
        X = torch.tensor(rows[FEATS].values, device=dev)
        y = torch.tensor(rows[YCOL].values, device=dev)
        tgt = true_grads(gmodel, X, y)
        rec = recover_rows(analytic_candidates(tgt))
        Xn = X.cpu().numpy()

        if len(rec):
            D = np.linalg.norm(rec[:, None, :] - Xn[None, :, :], axis=-1)
            correct = D.min(1) < 1e-2                      # matches a real row almost exactly
            hit = np.unique(D.argmin(1)[correct])
            pos_err = np.linalg.norm(unscale(torch.tensor(rec[correct]))[:, POS].numpy()
                                     - unscale(X[hit if len(hit) else []] if False else X)[D.argmin(1)[correct]][:, POS].numpy(), axis=1) if correct.any() else [np.nan]
        else:
            correct, hit, pos_err = np.array([], bool), [], [np.nan]

        rx_true = rows[['rx_x', 'rx_y']].values.mean(0)
        loc = (np.linalg.norm(locate_client(torch.tensor(rec)) - rx_true) if len(rec) >= 3 else np.nan)
        results4.append(dict(B=B, client=cid,
                             rows_claimed=len(rec),
                             precision=correct.mean() if len(rec) else np.nan,
                             frac_batch_exact=len(hit) / B,
                             pos_err_m=np.nanmedian(pos_err),
                             loc_err_m=loc))
    print(f"B={B} done ({time.time()-t0:.0f}s)")

res4 = pd.DataFrame(results4)
print(res4.groupby('B')[['rows_claimed', 'precision', 'frac_batch_exact', 'pos_err_m', 'loc_err_m']].median().round(3))
res4.to_csv(f"{OUT}/inversion_analytic_v1.csv", index=False)

B=2 done (0s)
B=4 done (0s)


/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err

B=8 done (0s)
B=16 done (1s)


/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err

B=32 done (1s)
B=64 done (1s)
    rows_claimed  precision  frac_batch_exact  pos_err_m  loc_err_m
B                                                                  
2            2.0        1.0             1.000        0.0      1.841
4            3.0        1.0             0.750        0.0      8.093
8            1.5        1.0             0.125        0.0     15.258
16           0.0        1.0             0.000        0.0    166.868
32           0.0        NaN             0.000        NaN        NaN
64           0.0        NaN             0.000        NaN        NaN


/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),
/tmp/ipykernel_2524/319579829.py:43: RuntimeWarning: All-NaN slice encountered
  pos_err_m=np.nanmedian(pos_err),


In [33]:
import warnings; warnings.filterwarnings('ignore', category=RuntimeWarning)

def train_fedavg(width, rounds=50, seed=0):
    random.seed(seed); torch.manual_seed(seed)
    m = MLP(N_FEAT, h=width, c=N_CLS).to(dev)
    for _ in range(rounds):
        sel = random.sample(cids, K)
        m.load_state_dict(fedavg([local_train(m, clients[c]['Xtr'], clients[c]['ytr']) for c in sel]))
    return m

def analytic_eval(model, B, cid, seed):
    rows = sample_window(cid, B, seed)
    X = torch.tensor(rows[FEATS].values, device=dev)
    y = torch.tensor(rows[YCOL].values, device=dev)
    rec = recover_rows(analytic_candidates(true_grads(model, X, y)))
    Xn, Xu = X.cpu().numpy(), unscale(X).numpy()
    frac, prec, perr = 0.0, np.nan, np.nan
    if len(rec):
        D = np.linalg.norm(rec[:, None, :] - Xn[None, :, :], axis=-1)
        nearest, correct = D.argmin(1), D.min(1) < 1e-2
        prec = correct.mean()
        frac = len(np.unique(nearest[correct])) / B
        if correct.any():
            P = unscale(torch.tensor(rec[correct]))[:, POS].numpy()
            perr = np.linalg.norm(P - Xu[nearest[correct]][:, POS], axis=1).mean()
    rx_true = rows[['rx_x', 'rx_y']].values.mean(0)
    loc = np.linalg.norm(locate_client(torch.tensor(rec)) - rx_true) if len(rec) >= 3 else np.nan
    return dict(frac_batch_exact=frac, precision=prec, pos_err_m=perr, loc_err_m=loc, rows_claimed=len(rec))

results5 = []
rng3 = np.random.default_rng(2)
eval_cids = rng3.choice(cids, 20, replace=False)
for width in [64, 256, 1024]:
    t0 = time.time()
    m = train_fedavg(width)
    m.eval()
    acc, f1 = evaluate(m, Xte_all, yte_all)
    print(f"width={width}: FedAvg macroF1={f1:.4f} (trained in {time.time()-t0:.0f}s)")
    for B in [4, 8, 16, 32, 64, 128]:
        for cid in eval_cids:
            r = analytic_eval(m, B, cid, seed=int(cid) % 1000 + B)
            results5.append(dict(width=width, B=B, client=cid, f1=f1, **r))

res5 = pd.DataFrame(results5)
summary = res5.groupby(['width', 'B'])[['frac_batch_exact', 'precision', 'pos_err_m',
                                        'loc_err_m', 'rows_claimed']].median().round(3)
print(summary)
res5.to_csv(f"{OUT}/inversion_analytic_width_sweep.csv", index=False)

width=64: FedAvg macroF1=0.9848 (trained in 12s)
width=256: FedAvg macroF1=0.9859 (trained in 12s)
width=1024: FedAvg macroF1=0.9860 (trained in 12s)
           frac_batch_exact  precision  pos_err_m  loc_err_m  rows_claimed
width B                                                                   
64    4               0.750      1.000      0.000      5.982           3.0
      8               0.125      1.000      0.000      4.946           1.0
      16              0.000      0.750      0.000        NaN           0.0
      32              0.000      0.000        NaN        NaN           0.0
      64              0.000        NaN        NaN        NaN           0.0
      128             0.000        NaN        NaN        NaN           0.0
256   4               1.000      0.667      0.000      8.983           6.0
      8               0.500      0.917      0.000      7.350           6.0
      16              0.062      1.000      0.000      9.159           1.0
      32              0.0

OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/capstone/outputs'

In [36]:
!cd /content && zip -r outputs_backup.zip outputs
from google.colab import files
files.download('/content/outputs_backup.zip')

  adding: outputs/ (stored 0%)
  adding: outputs/inversion_fedsgd_v1.csv (deflated 48%)
  adding: outputs/inversion_fedsgd_v2.csv (deflated 55%)
  adding: outputs/inversion_analytic_v1.csv (deflated 67%)
  adding: outputs/inversion_analytic_width_sweep.csv (deflated 76%)
  adding: outputs/inversion_fedsgd_v3.csv (deflated 47%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
def recover_rows(C, tol=1e-4, min_agree=3):
    used = np.zeros(len(C), bool); rec = []
    for i in range(len(C)):
        if used[i]: continue
        close = np.where(np.linalg.norm(C - C[i], axis=1) < tol)[0]
        used[close] = True
        if len(close) >= min_agree: rec.append(C[close].mean(0))
    return np.array(rec)

def score(rec, X, rows, thresh):
    Xn, Xu = X.cpu().numpy(), unscale(X).numpy()
    out = dict(rows_claimed=len(rec), precision=np.nan, frac_exact=0.0, pos_err_m=np.nan, loc_err_m=np.nan)
    if len(rec):
        D = np.linalg.norm(rec[:, None, :] - Xn[None, :, :], axis=-1)
        nearest, correct = D.argmin(1), D.min(1) < thresh
        out['precision'] = correct.mean()
        out['frac_exact'] = len(np.unique(nearest[correct])) / len(X)
        if correct.any():
            P = unscale(torch.tensor(rec[correct]))[:, POS].numpy()
            out['pos_err_m'] = np.linalg.norm(P - Xu[nearest[correct]][:, POS], axis=1).mean()
        if len(rec) >= 3:
            out['loc_err_m'] = np.linalg.norm(locate_client(torch.tensor(rec)) - rows[['rx_x', 'rx_y']].values.mean(0))
    return out

model = m                     # the width-1024 model from Cell 13
results6 = []
for N in [8, 16, 32, 64, 128, 256]:          # messages received since the vehicle's last round
    for cid in eval_cids:
        rows = sample_window(cid, N, seed=int(cid) % 1000 + N)
        X = torch.tensor(rows[FEATS].values, device=dev)
        y = torch.tensor(rows[YCOL].values, device=dev)

        # (a) single gradient over all N messages (FedSGD), with the fixed precision filter
        tgt = true_grads(model, X, y)
        C = (tgt[0][tgt[1].abs() > 1e-10] / tgt[1][tgt[1].abs() > 1e-10, None]).cpu().numpy()
        results6.append(dict(setting='FedSGD', N=N, client=cid, **score(recover_rows(C), X, rows, 1e-2)))

        # (b) real FedAvg update: 1 local epoch, batch 64, SGD lr 0.05, same as FL training
        before = copy.deepcopy(model.state_dict())
        after, _ = local_train(model, X, y)
        dW = before['net.0.weight'] - after['net.0.weight']
        db = before['net.0.bias'] - after['net.0.bias']
        mask = db.abs() > 1e-10
        C = (dW[mask] / db[mask, None]).cpu().numpy()
        results6.append(dict(setting='FedAvg', N=N, client=cid,
                             **score(recover_rows(C, tol=1e-3), X, rows, 5e-2)))
    print(f"N={N} done")

res6 = pd.DataFrame(results6)
print(res6.groupby(['setting', 'N'])[['rows_claimed', 'precision', 'frac_exact',
                                      'pos_err_m', 'loc_err_m']].median().round(3))
res6.to_csv(f"{OUT}/inversion_fedavg_vs_fedsgd.csv", index=False)

N=8 done
N=16 done
N=32 done
N=64 done
N=128 done


ValueError: high <= 0

In [38]:
def sample_window(cid, B, seed):
    g = train_rows[train_rows[R] == cid]
    if len(g) <= B:
        return None                                   # vehicle doesn't have enough messages
    start = np.random.default_rng(seed).integers(0, len(g) - B)
    return g.iloc[start:start + B]

results6 = [r for r in results6 if r['N'] != 256]     # drop any partial N=256 rows
N, skipped = 256, 0
for cid in eval_cids:
    rows = sample_window(cid, N, seed=int(cid) % 1000 + N)
    if rows is None:
        skipped += 1; continue
    X = torch.tensor(rows[FEATS].values, device=dev)
    y = torch.tensor(rows[YCOL].values, device=dev)

    tgt = true_grads(model, X, y)
    keep = tgt[1].abs() > 1e-10
    C = (tgt[0][keep] / tgt[1][keep, None]).cpu().numpy()
    results6.append(dict(setting='FedSGD', N=N, client=cid, **score(recover_rows(C), X, rows, 1e-2)))

    before = copy.deepcopy(model.state_dict())
    after, _ = local_train(model, X, y)
    dW = before['net.0.weight'] - after['net.0.weight']
    db = before['net.0.bias'] - after['net.0.bias']
    keep = db.abs() > 1e-10
    C = (dW[keep] / db[keep, None]).cpu().numpy()
    results6.append(dict(setting='FedAvg', N=N, client=cid, **score(recover_rows(C, tol=1e-3), X, rows, 5e-2)))
print(f"N=256: ran {len(eval_cids) - skipped} vehicles, skipped {skipped} with too few messages")

res6 = pd.DataFrame(results6)
print(res6.groupby(['setting', 'N'])[['rows_claimed', 'precision', 'frac_exact',
                                      'pos_err_m', 'loc_err_m']].median().round(3))
print("\nVehicles per setting:\n", res6.groupby(['setting', 'N']).size().unstack(0))
res6.to_csv(f"{OUT}/inversion_fedavg_vs_fedsgd.csv", index=False)

N=256: ran 18 vehicles, skipped 2 with too few messages
             rows_claimed  precision  frac_exact  pos_err_m  loc_err_m
setting N                                                             
FedAvg  8             0.0        1.0       0.000      0.391    183.379
        16            0.0        0.9       0.000      0.293    245.085
        32            0.0        0.0       0.000      0.300     94.583
        64            0.0        0.0       0.000        NaN        NaN
        128           0.0        0.0       0.000        NaN        NaN
        256           0.0        NaN       0.000        NaN        NaN
FedSGD  8             6.0        1.0       0.750      0.000      3.564
        16            2.5        1.0       0.156      0.000      7.890
        32            0.0        1.0       0.000      0.000     40.887
        64            0.0        1.0       0.000      0.000        NaN
        128           0.0        1.0       0.000      0.000        NaN
        256          

OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/capstone/outputs'

In [40]:
LOCAL_OUT = '/content/outputs'; os.makedirs(LOCAL_OUT, exist_ok=True)

def fedavg_delta(gmodel, X, y, dtype, seed=0):
    """One client's FedAvg round (1 local epoch, batch 64, SGD lr 0.05). Returns first-layer weight/bias change."""
    torch.manual_seed(seed)
    mm = copy.deepcopy(gmodel).to(dtype); mm.train()
    lf = nn.CrossEntropyLoss(weight=w.to(dtype))
    opt = torch.optim.SGD(mm.parameters(), lr=LR)
    X, y = X.to(dev, dtype), y.to(dev)
    W0 = mm.net[0].weight.detach().clone(); b0 = mm.net[0].bias.detach().clone()
    for _ in range(LOCAL_EPOCHS):
        perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), BS):
            idx = perm[i:i + BS]
            opt.zero_grad(); lf(mm(X[idx]), y[idx]).backward(); opt.step()
    return W0 - mm.net[0].weight.detach(), b0 - mm.net[0].bias.detach()

configs = [  # (label, dtype, recovery tolerance)
    ('float32_tol1e-3', torch.float32, 1e-3),
    ('float32_tol1e-2', torch.float32, 1e-2),
    ('float64_tol1e-4', torch.float64, 1e-4),
]

results7 = []
for N in [8, 16, 32, 64, 128, 256]:
    for cid in eval_cids:
        rows = sample_window(cid, N, seed=int(cid) % 1000 + N)
        if rows is None:
            continue
        X = torch.tensor(rows[FEATS].values, device=dev)
        y = torch.tensor(rows[YCOL].values, device=dev)
        for label, dtype, tol in configs:
            dW, db = fedavg_delta(m, X, y, dtype)
            keep = db.abs() > (1e-12 if dtype == torch.float64 else 1e-10)
            C = (dW[keep] / db[keep, None]).double().cpu().numpy()
            results7.append(dict(config=label, N=N, steps=int(np.ceil(N / BS)), client=cid,
                                 **score(recover_rows(C, tol=tol), X, rows, 1e-2)))
    print(f"N={N} done")

res7 = pd.DataFrame(results7)
print(res7.groupby(['config', 'N'])[['steps', 'rows_claimed', 'precision', 'frac_exact',
                                     'pos_err_m', 'loc_err_m']].median().round(3))
res7.to_csv(f"{LOCAL_OUT}/fedavg_precision_check.csv", index=False)

N=8 done
N=16 done
N=32 done
N=64 done
N=128 done
N=256 done
                     steps  rows_claimed  precision  frac_exact  pos_err_m  \
config          N                                                            
float32_tol1e-2 8      1.0           3.5      0.122       0.125      0.442   
                16     1.0          13.0      0.071       0.031      0.250   
                32     1.0           2.5      0.029       0.000      0.271   
                64     1.0           2.5      0.000       0.000      0.159   
                128    2.0           0.5      0.000       0.000        NaN   
                256    4.0           0.0      0.000       0.000      0.275   
float32_tol1e-3 8      1.0           0.0      0.350       0.000      0.164   
                16     1.0           0.0      0.600       0.000      0.293   
                32     1.0           0.0      0.000       0.000      0.300   
                64     1.0           0.0      0.000       0.000        NaN   
   

In [41]:
def fedavg_delta(gmodel, X, y, dtype, seed=0, bs=None):
    bs = bs or BS
    torch.manual_seed(seed)
    mm = copy.deepcopy(gmodel).to(dtype); mm.train()
    lf = nn.CrossEntropyLoss(weight=w.to(dtype))
    opt = torch.optim.SGD(mm.parameters(), lr=LR)
    X, y = X.to(dev, dtype), y.to(dev)
    W0 = mm.net[0].weight.detach().clone(); b0 = mm.net[0].bias.detach().clone()
    for _ in range(LOCAL_EPOCHS):
        perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), bs):
            idx = perm[i:i + bs]
            opt.zero_grad(); lf(mm(X[idx]), y[idx]).backward(); opt.step()
    return W0 - mm.net[0].weight.detach(), b0 - mm.net[0].bias.detach()

def attack_delta(model, rows, dtype, tol, bs=None):
    X = torch.tensor(rows[FEATS].values, device=dev)
    y = torch.tensor(rows[YCOL].values, device=dev)
    dW, db = fedavg_delta(model, X, y, dtype, bs=bs)
    keep = db.abs() > (1e-12 if dtype == torch.float64 else 1e-10)
    C = (dW[keep] / db[keep, None]).double().cpu().numpy()
    return score(recover_rows(C, tol=tol), X, rows, 1e-2)

# ---- A: train a width-1024 model with FedAvg, keeping snapshots from early rounds ----
random.seed(0); torch.manual_seed(0)
mA = MLP(N_FEAT, h=1024, c=N_CLS).to(dev)
snaps = {0: copy.deepcopy(mA.state_dict())}
for r in range(1, 51):
    sel = random.sample(cids, K)
    mA.load_state_dict(fedavg([local_train(mA, clients[c]['Xtr'], clients[c]['ytr']) for c in sel]))
    if r in (1, 5, 10, 25, 50):
        snaps[r] = copy.deepcopy(mA.state_dict())

resA = []
for rnd, sd_ in snaps.items():
    victim = MLP(N_FEAT, h=1024, c=N_CLS).to(dev); victim.load_state_dict(sd_)
    for N in [8, 16, 32]:
        for cid in eval_cids:
            rows = sample_window(cid, N, seed=int(cid) % 1000 + N)
            if rows is None: continue
            for label, dtype, tol in [('float32', torch.float32, 1e-3), ('float64', torch.float64, 1e-4)]:
                resA.append(dict(round=rnd, N=N, dtype=label, client=cid, **attack_delta(victim, rows, dtype, tol)))
    print(f"round {rnd} done")
resA = pd.DataFrame(resA)
print("\nA) Leakage by training round")
print(resA.groupby(['dtype', 'N', 'round'])[['frac_exact', 'precision', 'loc_err_m']].median().round(3).unstack('dtype'))

# ---- B: same messages, split into more local steps (float64, so precision isn't a factor) ----
resB = []
for N, batch_sizes in [(16, [16, 8, 4, 2]), (32, [32, 16, 8, 4])]:
    for bs in batch_sizes:
        for cid in eval_cids:
            rows = sample_window(cid, N, seed=int(cid) % 1000 + N)
            if rows is None: continue
            resB.append(dict(N=N, local_batch=bs, steps=N // bs, client=cid,
                             **attack_delta(m, rows, torch.float64, 1e-4, bs=bs)))
resB = pd.DataFrame(resB)
print("\nB) Effect of multiple local steps")
print(resB.groupby(['N', 'steps'])[['frac_exact', 'precision', 'loc_err_m']].median().round(3))

resA.to_csv(f"{LOCAL_OUT}/leakage_by_round.csv", index=False)
resB.to_csv(f"{LOCAL_OUT}/leakage_by_local_steps.csv", index=False)

round 0 done
round 1 done
round 5 done
round 10 done
round 25 done
round 50 done

A) Leakage by training round
         frac_exact         precision         loc_err_m        
dtype       float32 float64   float32 float64   float32 float64
N  round                                                       
8  0          0.500   0.750     1.000     1.0     4.350   3.388
   1          0.375   0.750     1.000     1.0     4.436   3.388
   5          0.125   0.750     1.000     1.0     2.938   3.410
   10         0.062   0.750     1.000     1.0    67.160   3.410
   25         0.000   0.750     0.945     1.0    60.358   3.410
   50         0.000   0.750     0.350     1.0   184.080   3.564
16 0          0.000   0.188     1.000     1.0   214.819   9.153
   1          0.031   0.188     1.000     1.0   723.276   9.153
   5          0.000   0.188     1.000     1.0   192.002   6.176
   10         0.000   0.188     1.000     1.0   227.314   6.176
   25         0.000   0.188     0.714     1.0   138.169  

In [43]:
pos_lo = train_rows[['pos_x', 'pos_y']].min().values
pos_hi = train_rows[['pos_x', 'pos_y']].max().values

def coarse_locate(C):
    """Locate the client from ALL first-layer candidates (no exact-recovery filtering)."""
    U = unscale(torch.tensor(C)).numpy()
    S = U[:, POS]
    dist = np.expm1(U[:, RXD]).clip(0, 2000)
    lo_u = scaler.mean_[POS] + pos_lo * scaler.scale_[POS]      # map bounds in metres
    hi_u = scaler.mean_[POS] + pos_hi * scaler.scale_[POS]
    ok = np.all(np.isfinite(S), axis=1) & np.all((S >= lo_u) & (S <= hi_u), axis=1)
    S, dist = S[ok], dist[ok]
    if len(S) == 0:
        return None, None, 0
    centroid = np.median(S, axis=0)
    tri = centroid
    if len(S) >= 3:
        f = lambda r: np.linalg.norm(S - r, axis=1) - dist
        tri = least_squares(f, centroid, loss='soft_l1', f_scale=20).x
    return centroid, tri, len(S)

resC = []
for rnd in [0, 5, 50]:
    victim = MLP(N_FEAT, h=1024, c=N_CLS).to(dev); victim.load_state_dict(snaps[rnd])
    for N in [8, 32, 128, 256, 512]:
        for cid in eval_cids:
            rows = sample_window(cid, N, seed=int(cid) % 1000 + N)
            if rows is None: continue
            X = torch.tensor(rows[FEATS].values, device=dev)
            y = torch.tensor(rows[YCOL].values, device=dev)
            dW, db = fedavg_delta(victim, X, y, torch.float32)          # realistic: float32, batch 64
            keep = db.abs() > 1e-10
            C = (dW[keep] / db[keep, None]).double().cpu().numpy()
            cen, tri, n_used = coarse_locate(C)
            rx_true = rows[['rx_x', 'rx_y']].values.mean(0)
            resC.append(dict(round=rnd, N=N, client=cid, candidates_used=n_used,
                             window_s=rows['rcvTime'].max() - rows['rcvTime'].min(),
                             loc_centroid_m=np.linalg.norm(cen - rx_true) if cen is not None else np.nan,
                             loc_trilat_m=np.linalg.norm(tri - rx_true) if tri is not None else np.nan,
                             baseline_m=np.linalg.norm(pos_mean - rx_true)))
    print(f"round {rnd} done")

resC = pd.DataFrame(resC)
print(resC.groupby(['round', 'N'])[['window_s', 'candidates_used', 'loc_centroid_m',
                                    'loc_trilat_m', 'baseline_m']].median().round(1))
print("\nVehicles per N:", resC.groupby('N').client.nunique().to_dict())
resC.to_csv(f"{LOCAL_OUT}/coarse_localization.csv", index=False)

round 0 done
round 5 done
round 50 done
           window_s  candidates_used  loc_centroid_m  loc_trilat_m  baseline_m
round N                                                                       
0     8         1.0            676.0           190.0          69.3  490.700012
      32        3.6            659.0           130.3          96.4  471.399994
      128      14.6            607.0           145.1         109.2  453.399994
      256      34.2            568.0           183.7         126.0  447.200012
      512      51.3            528.0           199.9         142.8  388.399994
5     8         1.0            739.0           174.7          61.1  490.700012
      32        3.6            724.5           150.1          88.8  471.399994
      128      14.6            738.0           161.2         183.3  453.399994
      256      34.2            648.5           186.7         155.5  447.200012
      512      51.3            464.0           131.1         159.2  388.399994
50    8     

In [44]:
own_t = own[['vehicle', 'rcvTime', 'pos_x', 'pos_y']]

def traffic_baseline(rows, k=200, seed=0):
    """Median distance from the true location to random OTHER vehicles on the road at the same time."""
    t0, t1 = rows['rcvTime'].min() - 5, rows['rcvTime'].max() + 5
    me = rows[R].iloc[0]
    pool = own_t[(own_t.rcvTime >= t0) & (own_t.rcvTime <= t1) & (own_t.vehicle != me)]
    if len(pool) == 0:
        return np.nan
    rx = rows[['rx_x', 'rx_y']].values.mean(0)
    s = pool.sample(min(k, len(pool)), random_state=seed)[['pos_x', 'pos_y']].values
    return np.median(np.linalg.norm(s - rx, axis=1))

tb = {}
for (N, cid) in resC[['N', 'client']].drop_duplicates().itertuples(index=False):
    rows = sample_window(cid, N, seed=int(cid) % 1000 + N)   # same windows as Cell 19
    if rows is not None:
        tb[(N, cid)] = traffic_baseline(rows)
resC['traffic_baseline_m'] = [tb.get((n, c), np.nan) for n, c in zip(resC.N, resC.client)]

print(resC.groupby(['round', 'N'])[['loc_trilat_m', 'baseline_m', 'traffic_baseline_m']].median().round(1))
resC.to_csv(f"{LOCAL_OUT}/coarse_localization.csv", index=False)

           loc_trilat_m  baseline_m  traffic_baseline_m
round N                                                
0     8            69.3  490.700012          659.599976
      32           96.4  471.399994          607.299988
      128         109.2  453.399994          600.200012
      256         126.0  447.200012          609.900024
      512         142.8  388.399994          502.799988
5     8            61.1  490.700012          659.599976
      32           88.8  471.399994          607.299988
      128         183.3  453.399994          600.200012
      256         155.5  447.200012          609.900024
      512         159.2  388.399994          502.799988
50    8            60.3  490.700012          659.599976
      32           73.2  471.399994          607.299988
      128         145.9  453.399994          600.200012
      256          96.4  447.200012          609.900024
      512         151.5  388.399994          502.799988


In [45]:
!cd /content && zip -r outputs_backup.zip outputs
from google.colab import files
files.download('/content/outputs_backup.zip')

updating: outputs/ (stored 0%)
updating: outputs/inversion_fedsgd_v1.csv (deflated 48%)
updating: outputs/inversion_fedsgd_v2.csv (deflated 55%)
updating: outputs/inversion_analytic_v1.csv (deflated 67%)
updating: outputs/inversion_analytic_width_sweep.csv (deflated 76%)
updating: outputs/inversion_fedsgd_v3.csv (deflated 47%)
  adding: outputs/coarse_localization.csv (deflated 60%)
  adding: outputs/leakage_by_local_steps.csv (deflated 67%)
  adding: outputs/leakage_by_round.csv (deflated 72%)
  adding: outputs/fedavg_precision_check.csv (deflated 75%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>